# Chunk ID and Section Audit for Retrieval

This notebook audits retrieval in three ways:

1. Check whether `relevant_chunk_ids` from evaluation data are actually retrieved.
2. Section-based chunk ID audit: for each section, verify returned chunk IDs and whether metadata maps to the expected section.
3. Guide-question audit: pick a question from the guide and show which chunk IDs are extracted, from which section, and what content is returned.

In [1]:
from __future__ import annotations

import json
import sys
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

try:
    import pandas as pd
except Exception:
    pd = None

cwd = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "backend" / "rag").exists() and (candidate / "backend" / "evaluation").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate project root containing backend/rag and backend/evaluation.")

BACKEND_DIR = PROJECT_ROOT / "backend"
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from rag.retrieval import RetrievalPipeline

DATASET_PATH = BACKEND_DIR / "evaluation" / "dataset" / "qa_pairs.json"
OUT_DIR = BACKEND_DIR / "evaluation" / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

with DATASET_PATH.open("r", encoding="utf-8") as f:
    qa_pairs = json.load(f)

valid_rows: list[dict[str, Any]] = []
for row in qa_pairs:
    relevant_ids = [str(x) for x in (row.get("relevant_chunk_ids") or [])]
    if not relevant_ids:
        continue
    if any("FILL IN" in rid for rid in relevant_ids):
        continue
    if "FILL IN" in str(row.get("question", "")):
        continue
    valid_rows.append(row)

pipeline = RetrievalPipeline()

print(f"Project root: {PROJECT_ROOT}")
print(f"Loaded QA rows: {len(qa_pairs)} | Valid rows: {len(valid_rows)}")
print(f"Output dir: {OUT_DIR}")

Configuration loaded:
  - API Host: 0.0.0.0:8000
  - Upload Directory: uploads
  - Max File Size: 50 MB
  - Groq API Configured: True
  - Qdrant Configured: True
  - Qdrant Collection: research_papers
  - LangSmith Tracing: Enabled
Project root: /home/aman/storage/Python/Projects/Research Paper Assistant
Loaded QA rows: 44 | Valid rows: 44
Output dir: /home/aman/storage/Python/Projects/Research Paper Assistant/backend/evaluation/results


In [2]:
TOP_K_CANDIDATES = 20
TOP_N_FINAL = 10


def normalize_query(q: str) -> str:
    return " ".join(str(q).strip().split())


def get_chunk_id(meta: dict[str, Any]) -> str | None:
    cid = meta.get("_id") or meta.get("chunk_id")
    return str(cid) if cid else None


def preview_text(text: str, max_chars: int = 220) -> str:
    s = " ".join((text or "").split())
    return s if len(s) <= max_chars else s[:max_chars].rstrip() + "..."


def run_section_scoped(query: str, document_id: str, section_id: str, rerank: bool = True) -> list[Any]:
    return pipeline.retrieve_with_section_scope(
        query=query,
        section_id=section_id,
        document_id=document_id,
        top_k=TOP_K_CANDIDATES,
        top_n=TOP_N_FINAL,
        rerank=rerank,
    )


def to_rows(results: list[Any], target_section_id: str) -> list[dict[str, Any]]:
    rows = []
    for rank, r in enumerate(results, start=1):
        meta = r.metadata if isinstance(getattr(r, "metadata", None), dict) else {}
        chunk_id = get_chunk_id(meta)
        section_path_ids = meta.get("section_path_ids") or []
        if not isinstance(section_path_ids, list):
            section_path_ids = []
        rows.append(
            {
                "rank": rank,
                "chunk_id": chunk_id,
                "score": float(getattr(r, "score", 0.0)),
                "section_id_meta": str(meta.get("section_id") or ""),
                "section_title_meta": str(meta.get("section_title") or ""),
                "content_type": str(meta.get("content_type") or ""),
                "target_section_id": target_section_id,
                "is_exact_section_match": str(meta.get("section_id") or "") == target_section_id,
                "is_in_section_path": target_section_id in [str(x) for x in section_path_ids],
                "preview": preview_text(str(getattr(r, "content", "") or "")),
            }
        )
    return rows

## 1) Relevant Chunk IDs: Are they retrieved?

For each QA row, run section-scoped retrieval and check if any `relevant_chunk_ids` are present in retrieved chunk IDs.

In [3]:
qa_retrieval_rows: list[dict[str, Any]] = []

for i, row in enumerate(valid_rows, start=1):
    question = normalize_query(str(row.get("question", "")))
    document_id = str(row.get("document_id", ""))
    section_id = str(row.get("section_id", ""))
    section_title = str(row.get("section_title", ""))
    relevant_ids = {str(x) for x in row.get("relevant_chunk_ids", [])}

    results = run_section_scoped(question, document_id, section_id, rerank=True)
    retrieved_ids = []
    for r in results:
        meta = r.metadata if isinstance(getattr(r, "metadata", None), dict) else {}
        cid = get_chunk_id(meta)
        if cid:
            retrieved_ids.append(cid)

    matched_ids = sorted(relevant_ids.intersection(set(retrieved_ids)))
    qa_retrieval_rows.append(
        {
            "idx": i,
            "paper_type": row.get("paper_type", ""),
            "question_type": row.get("question_type", ""),
            "section_title": section_title,
            "question": question,
            "document_id": document_id,
            "section_id": section_id,
            "relevant_ids": sorted(relevant_ids),
            "retrieved_ids": retrieved_ids,
            "matched_relevant_ids": matched_ids,
            "is_relevant_extracted": len(matched_ids) > 0,
            "n_relevant": len(relevant_ids),
            "n_matched": len(matched_ids),
        }
    )

hit_rate = sum(1 for r in qa_retrieval_rows if r["is_relevant_extracted"]) / max(len(qa_retrieval_rows), 1)
print(f"Rows checked: {len(qa_retrieval_rows)}")
print(f"Relevant extracted hit rate: {hit_rate:.2%}")

if pd is not None:
    qa_df = pd.DataFrame(qa_retrieval_rows)
    display(qa_df[["idx", "paper_type", "question_type", "section_title", "is_relevant_extracted", "n_matched", "n_relevant"]].head(20))

RetrievalPipeline: BM25 encoder not found at /home/aman/storage/Python/Projects/Research Paper Assistant/output/30c88170-fd15-5486-bf70-bbab16747183_bm25.pkl; sparse search will be disabled for this document
/home/aman/storage/Python/Projects/Research Paper Assistant/env_research/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
HybridRetriever: hybrid search failed (CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 1.64 GiB of which 15.31 MiB is free. Process 353577 has 1.49 GiB memory in use. Including non-PyTorch memory, this process has 132.00 MiB memory in use. Of the allocated memory 74.81 MiB is allocated by PyTorch, and 5.19 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:Tr

Rows checked: 44
Relevant extracted hit rate: 0.00%


,idx,paper_type,question_type,section_title,is_relevant_extracted,n_matched,n_relevant
0,1,Theory,factual,Abstract,False,0,3
1,2,Theory,factual,Abstract,False,0,3
2,3,Theory,conceptual,Abstract,False,0,2
3,4,Theory,conceptual,Lower bound,False,0,3
4,5,Theory,factual,Lower bound,False,0,3
5,6,Theory,conceptual,Second result,False,0,3
6,7,Theory,factual,Second result,False,0,3
7,8,Theory,factual,"Peter-Weyl theorem and Schur-Weyl duality, and...",False,0,3
8,9,Theory,conceptual,"Peter-Weyl theorem and Schur-Weyl duality, and...",False,0,3
9,10,Applied,factual,Attention,False,0,3


## 2) Section-Based Chunk ID Audit

For each unique section in QA data, retrieve chunks and verify:
- whether chunk metadata `section_id` exactly matches target section
- whether target section appears in metadata `section_path_ids` (descendant-safe check)

In [ ]:
section_groups = {}
for row in valid_rows:
    key = (str(row.get("document_id", "")), str(row.get("section_id", "")), str(row.get("section_title", "")))
    section_groups.setdefault(key, []).append(row)

section_audit_rows: list[dict[str, Any]] = []
for (document_id, section_id, section_title), grouped_rows in section_groups.items():
    # Use a representative query from this section (first question) to fetch chunks.
    query = normalize_query(str(grouped_rows[0].get("question", section_title)))
    results = run_section_scoped(query, document_id, section_id, rerank=True)

    for row in to_rows(results, section_id):
        row["document_id"] = document_id
        row["section_title_target"] = section_title
        row["query_used"] = query
        section_audit_rows.append(row)

print(f"Unique sections audited: {len(section_groups)}")
print(f"Chunk rows audited: {len(section_audit_rows)}")

if pd is not None:
    section_df = pd.DataFrame(section_audit_rows)
    summary = (
        section_df.groupby(["document_id", "target_section_id", "section_title_target"], as_index=False)
        .agg(
            n_chunks=("chunk_id", "count"),
            exact_match_rate=("is_exact_section_match", "mean"),
            in_path_rate=("is_in_section_path", "mean"),
        )
        .sort_values(["in_path_rate", "exact_match_rate"], ascending=True)
    )
    display(summary.head(20))

    mismatches = section_df[~section_df["is_in_section_path"] & ~section_df["is_exact_section_match"]]
    print(f"Hard mismatches (not exact and not in section path): {len(mismatches)}")
    if len(mismatches) > 0:
        display(mismatches[["document_id", "target_section_id", "section_title_target", "chunk_id", "section_id_meta", "section_title_meta", "rank"]].head(20))

HybridRetriever: hybrid search failed (CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 1.64 GiB of which 15.31 MiB is free. Process 353577 has 1.49 GiB memory in use. Including non-PyTorch memory, this process has 132.00 MiB memory in use. Of the allocated memory 74.81 MiB is allocated by PyTorch, and 5.19 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)); retrying with sparse only
HybridRetriever: dense fallback also failed: CUDA out of memory. Tried to allocate 46.00 MiB. GPU 0 has a total capacity of 1.64 GiB of which 15.31 MiB is free. Process 353577 has 1.49 GiB memory in use. Including non-PyTorch memory, this process has 132.00 MiB memory in use. Of the allocated memory 74.81 MiB is allocated by PyTorch, and 5.19 M

Unique sections audited: 17
Chunk rows audited: 0


KeyError: 'document_id'

: 

## 3) Guide Question -> Retrieved Chunk IDs and Output

This picks a guide question, resolves section ID, retrieves chunks, and shows chunk ID + section metadata + output preview.

In [ ]:
def load_guide_steps_for_doc(document_id: str) -> list[dict[str, Any]]:
    guide_name = f"{document_id}_guide.json"
    candidates = [
        PROJECT_ROOT / "output" / guide_name,
        PROJECT_ROOT.parent / "output" / guide_name,
    ]
    guide_path = next((p for p in candidates if p.exists()), None)
    if guide_path is None:
        return []

    data = json.loads(guide_path.read_text(encoding="utf-8"))
    if isinstance(data, list):
        return [x for x in data if isinstance(x, dict)]
    if isinstance(data, dict):
        if isinstance(data.get("steps"), list):
            return [x for x in data["steps"] if isinstance(x, dict)]
        # pass-based schema
        collected = []
        for value in data.values():
            if isinstance(value, dict) and isinstance(value.get("steps"), list):
                collected.extend([x for x in value["steps"] if isinstance(x, dict)])
        return collected
    return []


def normalize_label(s: str) -> str:
    return " ".join(str(s).strip().lower().split())


# Build fallback mapping from QA dataset: (doc_id, normalized section title) -> section_id.
qa_section_lookup: dict[tuple[str, str], str] = {}
for row in valid_rows:
    doc = str(row.get("document_id", ""))
    sid = str(row.get("section_id", ""))
    st = normalize_label(str(row.get("section_title", "")))
    if doc and sid and st and (doc, st) not in qa_section_lookup:
        qa_section_lookup[(doc, st)] = sid


def resolve_section_id_from_label(document_id: str, section_label: str) -> tuple[str | None, str]:
    # First try direct retrieval metadata.
    results = pipeline.query(query=section_label, document_id=document_id, top_k=5, rerank=False)
    for r in results:
        meta = r.metadata if isinstance(getattr(r, "metadata", None), dict) else {}
        sid = str(meta.get("section_id") or "").strip()
        stitle = str(meta.get("section_title") or section_label).strip() or section_label
        if sid:
            return sid, stitle

    # Fallback: QA-based mapping by section title text.
    sid = qa_section_lookup.get((document_id, normalize_label(section_label)))
    if sid:
        return sid, section_label

    return None, section_label


# Pick one document and one guide question automatically.
selected_doc_id = valid_rows[0]["document_id"] if valid_rows else ""
steps = load_guide_steps_for_doc(selected_doc_id)

picked = None
for step in steps:
    questions = step.get("questions_to_answer") or []
    sections = step.get("section_to_read") or []
    if isinstance(questions, list) and isinstance(sections, list) and questions and sections:
        picked = {
            "guide_step_number": step.get("step_number"),
            "question": str(questions[0]),
            "section_label": str(sections[0]),
        }
        break

if picked is None:
    raise RuntimeError(f"No guide question/section pair found for document_id={selected_doc_id}")

resolved_section_id, resolved_section_title = resolve_section_id_from_label(selected_doc_id, picked["section_label"])
if not resolved_section_id:
    raise RuntimeError(f"Could not resolve section_id for guide section label: {picked['section_label']}")

guide_results = run_section_scoped(
    query=normalize_query(picked["question"]),
    document_id=selected_doc_id,
    section_id=resolved_section_id,
    rerank=True,
)

guide_rows = to_rows(guide_results, resolved_section_id)
for row in guide_rows:
    row["guide_step_number"] = picked["guide_step_number"]
    row["guide_question"] = picked["question"]
    row["guide_section_label"] = picked["section_label"]
    row["resolved_section_title"] = resolved_section_title
    row["document_id"] = selected_doc_id

print("Selected guide question audit:")
print(f"  document_id: {selected_doc_id}")
print(f"  step_number: {picked['guide_step_number']}")
print(f"  section_label: {picked['section_label']}")
print(f"  resolved_section_id: {resolved_section_id}")
print(f"  question: {picked['question']}")
print(f"  chunks returned: {len(guide_rows)}")

if pd is not None:
    guide_df = pd.DataFrame(guide_rows)
    display(guide_df[["rank", "chunk_id", "score", "section_id_meta", "section_title_meta", "is_exact_section_match", "is_in_section_path", "preview"]].head(10))
else:
    for row in guide_rows[:10]:
        print(f"[{row['rank']}] chunk_id={row['chunk_id']} | score={row['score']:.4f}")
        print(f"    section={row['section_title_meta']} ({row['section_id_meta']})")
        print(f"    preview={row['preview']}")

INFO:httpx:HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers/points/query "HTTP/1.1 200 OK"
INFO:rag.retrieval.search.hybrid_retriever:HybridRetriever: query='Abstract' → 5 results (doc_filter=30c88170-fd15-5486-bf70-bbab16747183)
INFO:httpx:HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://b9c1abff-fc0c-4fba-8838-f0890

Selected guide question audit:
  document_id: 30c88170-fd15-5486-bf70-bbab16747183
  step_number: 1
  section_label: Abstract
  resolved_section_id: 30c88170-fd15-5486-bf70-bbab16747183_section_0
  question: What is the main problem addressed in the paper?
  chunks returned: 10


,rank,chunk_id,score,section_id_meta,section_title_meta,is_exact_section_match,is_in_section_path,preview
0,1,2bbbf78b-3a30-526f-a22c-3d000aa050d1,0.001377,,,False,False,"For general functions f : U(d) →C, no efficien..."
1,2,bad4cb28-49f9-5bee-b779-b9ed4bc0ec8b,0.000673,,,False,False,1.3 Organization of the paper The remainder of...
2,3,cde46115-8a9d-5842-90cd-1e7bb8a1c9c0,0.000189,,,False,False,These observations support the wide applicabil...
3,4,684a677e-19ba-5811-836c-87ad8634b306,0.000189,,,False,False,These observations support the wide applicabil...
4,5,282bda96-eeb2-5e86-ac87-0ce39a08c9fa,0.000066,,,False,False,As representation theory provides powerful too...
5,6,08356e77-dee2-5db7-b006-025e02a3bd84,0.000062,,,False,False,"... ... an1B an2B annB     . Let N0 = {0,..."
6,7,a0b6afbd-c2ad-5b86-801e-e3a8d7543112,0.000040,,,False,False,"... ... an1B an2B annB     . Let N0 = {0,..."
7,8,44fdbe21-ace5-5090-b053-d494a3258a61,0.000034,,,False,False,for some matrix Af. The singular value decompo...
8,9,3d97ad35-2163-5585-849a-f084ea9b7911,0.000034,,,False,False,for some matrix Af. The singular value decompo...
9,10,b60e7098-8b01-573e-902c-d0f1b4de8b1d,0.000029,,,False,False,Proposition 1 shows that to get an upper bound...


In [ ]:
qa_out = OUT_DIR / "chunk_id_audit_qa_relevant_check.json"
section_out = OUT_DIR / "chunk_id_audit_section_check.json"
guide_out = OUT_DIR / "chunk_id_audit_guide_question_check.json"

qa_out.write_text(json.dumps(qa_retrieval_rows, indent=2, ensure_ascii=False), encoding="utf-8")
section_out.write_text(json.dumps(section_audit_rows, indent=2, ensure_ascii=False), encoding="utf-8")
guide_out.write_text(json.dumps(guide_rows, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"Saved: {qa_out}")
print(f"Saved: {section_out}")
print(f"Saved: {guide_out}")

if pd is not None:
    pd.DataFrame(qa_retrieval_rows).to_csv(OUT_DIR / "chunk_id_audit_qa_relevant_check.csv", index=False)
    pd.DataFrame(section_audit_rows).to_csv(OUT_DIR / "chunk_id_audit_section_check.csv", index=False)
    pd.DataFrame(guide_rows).to_csv(OUT_DIR / "chunk_id_audit_guide_question_check.csv", index=False)
    print("Saved CSV versions too.")

Saved: /home/aman/storage/Python/Projects/Research Paper Assistant/backend/evaluation/results/chunk_id_audit_qa_relevant_check.json
Saved: /home/aman/storage/Python/Projects/Research Paper Assistant/backend/evaluation/results/chunk_id_audit_section_check.json
Saved: /home/aman/storage/Python/Projects/Research Paper Assistant/backend/evaluation/results/chunk_id_audit_guide_question_check.json
Saved CSV versions too.


## 4) Chunk Size Audit Per Paper

This section measures chunk size for each chunk in each paper (`document_id`) by scrolling all indexed chunks from Qdrant.

Reported metrics:
- per-chunk size fields: character count, word count, and estimated token count
- per-paper summary: number of chunks, averages, medians, min/max, and p90 for each size metric
- overall summary across all papers

In [ ]:
from math import ceil


def estimate_tokens_from_words(word_count: int) -> int:
    # Rough heuristic for English technical text; keeps reporting simple and stable.
    return int(ceil(word_count * 1.3))


def iter_document_chunks(document_id: str, batch_size: int = 256):
    from qdrant_client.models import Filter, FieldCondition, MatchValue

    client = pipeline._get_store_manager().client
    collection = pipeline._get_store_manager().collection_name
    offset = None

    while True:
        points, next_offset = client.scroll(
            collection_name=collection,
            scroll_filter=Filter(
                must=[
                    FieldCondition(
                        key="document_id",
                        match=MatchValue(value=document_id),
                    )
                ]
            ),
            limit=batch_size,
            with_payload=True,
            with_vectors=False,
            offset=offset,
        )

        if not points:
            break

        for p in points:
            yield p.payload if isinstance(p.payload, dict) else {}

        offset = next_offset
        if offset is None:
            break


def payload_chunk_id(payload: dict[str, Any], fallback_idx: int) -> str:
    cid = payload.get("_id") or payload.get("chunk_id")
    return str(cid) if cid else f"missing_chunk_id_{fallback_idx}"


chunk_size_rows: list[dict[str, Any]] = []

document_ids = sorted({str(r.get("document_id", "")) for r in valid_rows if str(r.get("document_id", ""))})

for doc_id in document_ids:
    seen_chunk_ids: set[str] = set()
    fallback_idx = 0

    for payload in iter_document_chunks(doc_id):
        fallback_idx += 1
        chunk_id = payload_chunk_id(payload, fallback_idx)
        if chunk_id in seen_chunk_ids:
            continue
        seen_chunk_ids.add(chunk_id)

        content = str(payload.get("content") or "")
        words = content.split()
        word_count = len(words)
        char_count = len(content)

        chunk_size_rows.append(
            {
                "document_id": doc_id,
                "chunk_id": chunk_id,
                "section_id": str(payload.get("section_id") or ""),
                "section_title": str(payload.get("section_title") or ""),
                "content_type": str(payload.get("content_type") or ""),
                "chunk_level": str(payload.get("chunk_level") or ""),
                "char_count": char_count,
                "word_count": word_count,
                "est_token_count": estimate_tokens_from_words(word_count),
            }
        )

print(f"Documents audited: {len(document_ids)}")
print(f"Unique chunks audited: {len(chunk_size_rows)}")

if pd is not None and chunk_size_rows:
    chunk_size_df = pd.DataFrame(chunk_size_rows)

    per_paper_summary = (
        chunk_size_df.groupby("document_id", as_index=False)
        .agg(
            n_chunks=("chunk_id", "count"),
            avg_chars=("char_count", "mean"),
            median_chars=("char_count", "median"),
            min_chars=("char_count", "min"),
            max_chars=("char_count", "max"),
            p90_chars=("char_count", lambda s: s.quantile(0.90)),
            avg_words=("word_count", "mean"),
            median_words=("word_count", "median"),
            min_words=("word_count", "min"),
            max_words=("word_count", "max"),
            p90_words=("word_count", lambda s: s.quantile(0.90)),
            avg_est_tokens=("est_token_count", "mean"),
            median_est_tokens=("est_token_count", "median"),
            min_est_tokens=("est_token_count", "min"),
            max_est_tokens=("est_token_count", "max"),
            p90_est_tokens=("est_token_count", lambda s: s.quantile(0.90)),
        )
        .sort_values("avg_words", ascending=False)
    )

    overall_summary = pd.DataFrame(
        [
            {
                "n_documents": int(chunk_size_df["document_id"].nunique()),
                "n_chunks": int(len(chunk_size_df)),
                "avg_chars": float(chunk_size_df["char_count"].mean()),
                "median_chars": float(chunk_size_df["char_count"].median()),
                "avg_words": float(chunk_size_df["word_count"].mean()),
                "median_words": float(chunk_size_df["word_count"].median()),
                "avg_est_tokens": float(chunk_size_df["est_token_count"].mean()),
                "median_est_tokens": float(chunk_size_df["est_token_count"].median()),
            }
        ]
    )

    display(per_paper_summary)
    display(overall_summary)

    # Save outputs for reuse in external analysis.
    chunk_size_out_json = OUT_DIR / "chunk_size_audit_per_chunk.json"
    chunk_size_out_csv = OUT_DIR / "chunk_size_audit_per_chunk.csv"
    chunk_size_summary_csv = OUT_DIR / "chunk_size_audit_per_paper_summary.csv"

    chunk_size_out_json.write_text(json.dumps(chunk_size_rows, indent=2, ensure_ascii=False), encoding="utf-8")
    chunk_size_df.to_csv(chunk_size_out_csv, index=False)
    per_paper_summary.to_csv(chunk_size_summary_csv, index=False)

    print(f"Saved: {chunk_size_out_json}")
    print(f"Saved: {chunk_size_out_csv}")
    print(f"Saved: {chunk_size_summary_csv}")
elif pd is None:
    print("pandas is not available; install pandas to render summary tables.")
else:
    print("No chunks found for document IDs in the QA dataset.")

INFO:httpx:HTTP Request: POST https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers/points/scroll "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers/points/scroll "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers/points/scroll "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers/points/scroll "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers/points/scroll "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers/points

Documents audited: 3
Unique chunks audited: 1022


,document_id,n_chunks,avg_chars,median_chars,min_chars,max_chars,p90_chars,avg_words,median_words,min_words,max_words,p90_words,avg_est_tokens,median_est_tokens,min_est_tokens,max_est_tokens,p90_est_tokens
1,bd077a96-5a38-5281-993e-10cf869afcde,185,752.054054,568.0,47,1815,1565.0,116.383784,92.0,7,279,252.0,151.772973,120.0,10,363,328.0
2,e0960904-0d88-57cb-a52e-f60e01df2c7b,563,730.031972,549.0,27,1844,1480.0,106.644760,81.0,4,333,220.8,139.081705,106.0,6,433,287.6
0,30c88170-fd15-5486-bf70-bbab16747183,274,554.718978,417.0,54,1731,1179.4,97.135036,73.0,9,281,195.0,126.770073,95.0,12,366,254.0


,n_documents,n_chunks,avg_chars,median_chars,avg_words,median_words,avg_est_tokens,median_est_tokens
0,3,1022,687.016634,521.0,105.858121,81.0,138.078278,106.0


Saved: /home/aman/storage/Python/Projects/Research Paper Assistant/backend/evaluation/results/chunk_size_audit_per_chunk.json
Saved: /home/aman/storage/Python/Projects/Research Paper Assistant/backend/evaluation/results/chunk_size_audit_per_chunk.csv
Saved: /home/aman/storage/Python/Projects/Research Paper Assistant/backend/evaluation/results/chunk_size_audit_per_paper_summary.csv


In [ ]:
per_paper_summary


,document_id,n_chunks,avg_chars,median_chars,min_chars,max_chars,p90_chars,avg_words,median_words,min_words,max_words,p90_words,avg_est_tokens,median_est_tokens,min_est_tokens,max_est_tokens,p90_est_tokens
1,bd077a96-5a38-5281-993e-10cf869afcde,185,752.054054,568.0,47,1815,1565.0,116.383784,92.0,7,279,252.0,151.772973,120.0,10,363,328.0
2,e0960904-0d88-57cb-a52e-f60e01df2c7b,563,730.031972,549.0,27,1844,1480.0,106.644760,81.0,4,333,220.8,139.081705,106.0,6,433,287.6
0,30c88170-fd15-5486-bf70-bbab16747183,274,554.718978,417.0,54,1731,1179.4,97.135036,73.0,9,281,195.0,126.770073,95.0,12,366,254.0


In [ ]:
if pd is not None and chunk_size_rows:
    print("Chunk Size Summary Report")
    print("=" * 40)
    overall = overall_summary.iloc[0]
    print(f"Documents: {int(overall['n_documents'])}")
    print(f"Chunks: {int(overall['n_chunks'])}")
    print(f"Average chars/chunk: {overall['avg_chars']:.2f} | Median: {overall['median_chars']:.2f}")
    print(f"Average words/chunk: {overall['avg_words']:.2f} | Median: {overall['median_words']:.2f}")
    print(f"Average est tokens/chunk: {overall['avg_est_tokens']:.2f} | Median: {overall['median_est_tokens']:.2f}")
    print()
    print("Per-paper averages (sorted by avg words desc):")
    report_cols = ["document_id", "n_chunks", "avg_chars", "avg_words", "avg_est_tokens"]
    print(per_paper_summary[report_cols].to_string(index=False, justify="left", col_space=12))
else:
    print("Chunk size report unavailable (no rows or pandas not installed).")

Chunk Size Summary Report
Documents: 3
Chunks: 1022
Average chars/chunk: 687.02 | Median: 521.00
Average words/chunk: 105.86 | Median: 81.00
Average est tokens/chunk: 138.08 | Median: 106.00

Per-paper averages (sorted by avg words desc):
document_id                           n_chunks     avg_chars    avg_words    avg_est_tokens
bd077a96-5a38-5281-993e-10cf869afcde 185          752.054054   116.383784   151.772973     
e0960904-0d88-57cb-a52e-f60e01df2c7b 563          730.031972   106.644760   139.081705     
30c88170-fd15-5486-bf70-bbab16747183 274          554.718978    97.135036   126.770073     


In [ ]:
import numpy as np
tokens = [chunk['estimated_tokens'] for chunk in chunks]
print(np.percentile(tokens, [90, 95, 99]))

NameError: name 'chunks' is not defined

In [ ]:
import numpy as np

# Extract estimated token counts from the chunk size rows.
estimated_tokens = [row['est_token_count'] for row in chunk_size_rows]

# Calculate and print the 90th, 95th, and 99th percentiles to check the upper tail.
percentiles = np.percentile(estimated_tokens, [90, 95, 99])
print("Upper Tail Percentiles (Estimated Tokens):")
print(f"90th Percentile: {percentiles[0]:.2f}")
print(f"95th Percentile: {percentiles[1]:.2f}")
print(f"99th Percentile: {percentiles[2]:.2f}")

Upper Tail Percentiles (Estimated Tokens):
90th Percentile: 286.00
95th Percentile: 319.95
99th Percentile: 354.79


In [ ]:
# Configurable inputs: change these to test any section/document.
SECTION_TO_CHECK = "Introduction"
DOCUMENT_ID_TO_CHECK = valid_rows[0]["document_id"] if valid_rows else ""
QUERY_FOR_RETRIEVAL = f"Key points from {SECTION_TO_CHECK}"
TOP_K_FOR_CHECK = 20
TOP_N_FOR_CHECK = 20


def _norm(s: str) -> str:
    return " ".join(str(s).strip().lower().split())


if not DOCUMENT_ID_TO_CHECK:
    raise RuntimeError("No document_id available. Ensure valid_rows is populated first.")

# Build section lookup for the selected document from QA rows.
sections_for_doc = []
for r in valid_rows:
    if str(r.get("document_id", "")) != DOCUMENT_ID_TO_CHECK:
        continue
    sid = str(r.get("section_id", ""))
    stitle = str(r.get("section_title", ""))
    if sid and stitle:
        sections_for_doc.append((sid, stitle))

# Deduplicate while preserving order.
seen_pairs = set()
sections_for_doc_unique = []
for pair in sections_for_doc:
    if pair in seen_pairs:
        continue
    seen_pairs.add(pair)
    sections_for_doc_unique.append(pair)

if not sections_for_doc_unique:
    raise RuntimeError(f"No section metadata found in QA rows for document_id '{DOCUMENT_ID_TO_CHECK}'.")

section_label_norm = _norm(SECTION_TO_CHECK)

# 1) Exact normalized title match.
exact_pairs = [(sid, st) for sid, st in sections_for_doc_unique if _norm(st) == section_label_norm]

# 2) Partial title match fallback.
partial_pairs = [(sid, st) for sid, st in sections_for_doc_unique if section_label_norm in _norm(st)]

matched_pairs = exact_pairs or partial_pairs

if matched_pairs:
    SECTION_ID_TO_CHECK, matched_section_title = matched_pairs[0]
    match_mode = "exact" if exact_pairs else "partial"
else:
    # Graceful fallback so the cell remains runnable and exploratory.
    SECTION_ID_TO_CHECK, matched_section_title = sections_for_doc_unique[0]
    match_mode = "fallback_first_available"

# Retrieve chunks scoped to the chosen section, then inspect whether metadata aligns.
section_check_results = pipeline.retrieve_with_section_scope(
    query=QUERY_FOR_RETRIEVAL,
    section_id=SECTION_ID_TO_CHECK,
    document_id=DOCUMENT_ID_TO_CHECK,
    top_k=TOP_K_FOR_CHECK,
    top_n=TOP_N_FOR_CHECK,
    rerank=True,
)

section_chunk_rows: list[dict[str, Any]] = []
for rank, result in enumerate(section_check_results, start=1):
    meta = result.metadata if isinstance(getattr(result, "metadata", None), dict) else {}
    section_chunk_rows.append(
        {
            "rank": rank,
            "chunk_id": get_chunk_id(meta),
            "score": float(getattr(result, "score", 0.0)),
            "requested_section": SECTION_TO_CHECK,
            "resolved_section_title": matched_section_title,
            "requested_section_id": SECTION_ID_TO_CHECK,
            "match_mode": match_mode,
            "section_title_meta": str(meta.get("section_title") or ""),
            "section_id_meta": str(meta.get("section_id") or ""),
            "is_exact_section_title": _norm(str(meta.get("section_title") or "")) == _norm(matched_section_title),
            "is_exact_section_id": str(meta.get("section_id") or "") == SECTION_ID_TO_CHECK,
            "preview": preview_text(str(getattr(result, "content", "") or ""), max_chars=260),
        }
    )

available_titles = sorted({st for _, st in sections_for_doc_unique})

print(f"Document: {DOCUMENT_ID_TO_CHECK}")
print(f"Requested section label: {SECTION_TO_CHECK}")
print(f"Resolved section title: {matched_section_title}")
print(f"Resolved section_id: {SECTION_ID_TO_CHECK}")
print(f"Match mode: {match_mode}")
print(f"Chunks returned: {len(section_chunk_rows)}")

if match_mode == "fallback_first_available":
    print("No exact/partial title match found. Using first available section for this document.")

print("Available section titles for this document:")
for title in available_titles[:30]:
    print(f"- {title}")

if pd is not None:
    section_check_df = pd.DataFrame(section_chunk_rows)
    display(
        section_check_df[
            [
                "rank",
                "chunk_id",
                "score",
                "requested_section",
                "resolved_section_title",
                "section_title_meta",
                "section_id_meta",
                "is_exact_section_title",
                "is_exact_section_id",
                "match_mode",
                "preview",
            ]
        ]
    )

    print("\nMatch summary:")
    print(f"  Exact title matches: {int(section_check_df['is_exact_section_title'].sum())}/{len(section_check_df)}")
    print(f"  Exact section_id matches: {int(section_check_df['is_exact_section_id'].sum())}/{len(section_check_df)}")
else:
    for row in section_chunk_rows:
        print(
            f"[{row['rank']}] chunk_id={row['chunk_id']} | score={row['score']:.4f} | "
            f"section_title_meta='{row['section_title_meta']}' | section_id_meta='{row['section_id_meta']}'"
        )

INFO:httpx:HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers/points/query "HTTP/1.1 200 OK"
INFO:rag.retrieval.search.hybrid_retriever:HybridRetriever: query='Key points from Introduction' → 20 results (doc_filter=30c88170-fd15-5486-bf70-bbab16747183)
INFO:rag.retrieval.search.reranker:FlashRankReranker: reranked 20 → 12 results for query 'Key points from Introduction'


Document: 30c88170-fd15-5486-bf70-bbab16747183
Requested section label: Introduction
Resolved section title: Abstract
Resolved section_id: 30c88170-fd15-5486-bf70-bbab16747183_section_0
Match mode: fallback_first_available
Chunks returned: 12
No exact/partial title match found. Using first available section for this document.
Available section titles for this document:
- Abstract
- Lower bound
- Peter-Weyl theorem and Schur-Weyl duality, and their consequences
- Second result
- Upper bound


,rank,chunk_id,score,requested_section,resolved_section_title,section_title_meta,section_id_meta,is_exact_section_title,is_exact_section_id,match_mode,preview
0,1,f105ce13-c987-5568-9aaa-47f688cf9fdc,0.000131,Introduction,Abstract,,,False,False,fallback_first_available,Remarks on Theorem 1. We now give several rema...
1,2,738352ed-8c32-5154-8b81-91d8cb8b88c9,0.000131,Introduction,Abstract,,,False,False,fallback_first_available,Remarks on Theorem 1. We now give several rema...
2,3,a5e9c25a-f226-53ef-ab97-01224f2c24eb,0.000129,Introduction,Abstract,,,False,False,fallback_first_available,"However, the query complexity of the matrix el..."
3,4,49991b05-ecfe-5cfc-a881-7d8231ec94cb,0.000091,Introduction,Abstract,,,False,False,fallback_first_available,This works for any function f that is square-i...
4,5,bad4cb28-49f9-5bee-b779-b9ed4bc0ec8b,0.000038,Introduction,Abstract,,,False,False,fallback_first_available,1.3 Organization of the paper The remainder of...
5,6,4f2bc9d6-a0e1-55c4-a2d5-fb24af841ed2,0.000025,Introduction,Abstract,,,False,False,fallback_first_available,1.2 Proof techniques We now briefly explain ho...
6,7,08356e77-dee2-5db7-b006-025e02a3bd84,0.000024,Introduction,Abstract,,,False,False,fallback_first_available,"... ... an1B an2B annB     . Let N0 = {0,..."
7,8,3eb99ddf-fa54-5831-98a0-02fc209eb40f,0.000023,Introduction,Abstract,,,False,False,fallback_first_available,This implies that the number m of queries has ...
8,9,dd63d50a-8003-5b61-a767-f23c641fa02c,0.000020,Introduction,Abstract,,,False,False,fallback_first_available,Upper bounds. To construct the query algorithm...
9,10,ca57e8b0-7e65-588e-a7eb-67936b784b72,0.000020,Introduction,Abstract,,,False,False,fallback_first_available,Remarks on Theorem 1. We now give several rema...



Match summary:
  Exact title matches: 0/12
  Exact section_id matches: 0/12


: 